In [1]:
import geopandas as gpd

Read the shapefil using geopandas and using the encoding as "utf-8" to handle Norwegian characters correctly.

In [2]:
regions = gpd.read_file("../data/shapefiles/NVE_region.shp", encoding = 'utf-8')
# . means current directory, .. means parent directory

print("Coordinate Reference System:", regions.crs)  # Coordinate Reference System
print("Column names:", regions.columns.tolist())  # List of column names

regions

Coordinate Reference System: EPSG:25833
Column names: ['OBJECTID', 'REGION', 'REGIONNAVN', 'POSTNUMMER', 'POSTSTED', 'BESØKSADR', 'ADRESSE', 'geometry']


,OBJECTID,REGION,REGIONNAVN,POSTNUMMER,POSTSTED,BESØKSADR,ADRESSE,geometry
0,1.0,RV,Region Vest,6801,Førde,Naustdalsveien 1B,Postboks 53,"POLYGON ((193105.76 6929797.839, 191705.75 692..."
1,2.0,RM,Region Midt-Norge,7030,Trondheim,"Abelsgate 9, Trondheim",Abelsgate 9,"POLYGON ((391250 7388620, 392270 7386790, 3937..."
2,3.0,RN,Region Nord,8514,Narvik,Kongensgate 52-54,Kongensgate 52-54,"POLYGON ((1109096.413 7844647.502, 1105115.343..."
3,4.0,RØ,Region Øst,2307,Hamar,Vangsvegen 73,Postboks 4223,"POLYGON ((253748.22 6678130.91, 253645.58 6678..."
4,5.0,RS,Region Sør,3103,Tønsberg,Anton Jenssens gate 7,Postboks 2124,"POLYGON ((253748.22 6678130.91, 253917.63 6677..."


In [3]:
print(regions[["REGION", "REGIONNAVN"]].to_string(index=False))
print("Number of regions:", len(regions))

REGION        REGIONNAVN
    RV       Region Vest
    RM Region Midt-Norge
    RN       Region Nord
    RØ        Region Øst
    RS        Region Sør
Number of regions: 5


There are five administrative NVE regions in total

## Generate random points for each region 

to use for weather data download

In [7]:
import numpy as np
from shapely.geometry import Point

def sample_random_points(regions, n_points=1000, random_state=42):
    """
    Generate random points within each region identifier.
    """

    # Set the random seed for reproducibility
    rng = np.random.default_rng(random_state)
    records = []

    # Loop through each region and generate random points
    for _, region in regions.iterrows():

        polygon = region.geometry
        minx, miny, maxx, maxy = polygon.bounds

        count = 0

        # Generate random points until we reach the desired number of points for this region
        while count < n_points:

            x = rng.uniform(minx, maxx)
            y = rng.uniform(miny, maxy)

            point = Point(x, y)

            if polygon.contains(point):

                records.append({
                    "REGION": region["REGION"],
                    "geometry": point
                })

                count += 1

    points = gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs=regions.crs
    )

    # Convert projected coordinates to latitude/longitude
    points = points.to_crs(epsg=4326)

    points["longitude"] = points.geometry.x
    points["latitude"] = points.geometry.y

    return points


random_points = sample_random_points(
    regions,
    n_points=1000,
    random_state=42
)

print('Size of random_points for each region:')
print(random_points.groupby("REGION").size())
random_points

Size of random_points for each region:
REGION
RM    1000
RN    1000
RS    1000
RV    1000
RØ    1000
dtype: int64


,REGION,geometry,longitude,latitude
0,RV,POINT (8.57911 62.56125),8.579105,62.561251
1,RV,POINT (7.92645 62.8967),7.926451,62.896700
2,RV,POINT (4.71421 61.30104),4.714213,61.301037
3,RV,POINT (7.19869 63.01013),7.198689,63.010126
4,RV,POINT (6.70625 60.51347),6.706254,60.513469
...,...,...,...,...
4995,RS,POINT (6.64022 58.77349),6.640216,58.773493
4996,RS,POINT (7.46436 58.55155),7.464360,58.551554
4997,RS,POINT (6.5078 58.94622),6.507796,58.946224
4998,RS,POINT (7.63451 59.96484),7.634513,59.964839


## Get historical weather data from Open-Meteo (ERA5)

In [ ]:
import requests
import pandas as pd

def get_era5_weather(latitude, longitude, start_date, end_date):
    """
    Download hourly ERA5 reanalysis weather data
    for a single geographic location.
    """

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation",
        "models": "era5",
        "timezone": "UTC"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()["hourly"]

    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"], utc=True)

    return df

## Get ERA5-data based on region code

In [10]:
def get_region_weather(
    region,
    start_date,
    end_date,
    points_df=random_points,
    n_locations=1
):
    """
    Download hourly ERA5 weather data for locations
    sampled within a specified NVE region.

    """

    # Select locations belonging to the region
    region_points = points_df[
        points_df["REGION"] == region
    ]

    if region_points.empty:
        raise ValueError(f"Unknown region: {region}")

    if not 1 <= n_locations <= len(region_points):
        raise ValueError("Invalid number of locations")

    results = []

    for _, point in region_points.head(n_locations).iterrows():

        weather = get_era5_weather(
            latitude=point["latitude"],
            longitude=point["longitude"],
            start_date=start_date,
            end_date=end_date
        )

        weather["REGION"] = region
        weather["latitude"] = point["latitude"]
        weather["longitude"] = point["longitude"]

        results.append(weather)

    return pd.concat(results, ignore_index=True)

In [13]:
weather_rm = get_region_weather(
    region="RM", # Test with one of the regions (e.g., "RM" for Region Midt-Norge)
    start_date="2024-01-01",
    end_date="2024-12-31",
    n_locations=1
)

weather_rm

,time,temperature_2m,precipitation,REGION,latitude,longitude
0,2024-01-01 00:00:00+00:00,-3.9,0.0,RM,64.020179,10.35306
1,2024-01-01 01:00:00+00:00,-4.0,0.0,RM,64.020179,10.35306
2,2024-01-01 02:00:00+00:00,-3.9,0.0,RM,64.020179,10.35306
3,2024-01-01 03:00:00+00:00,-3.7,0.0,RM,64.020179,10.35306
4,2024-01-01 04:00:00+00:00,-3.6,0.0,RM,64.020179,10.35306
...,...,...,...,...,...,...
8779,2024-12-31 19:00:00+00:00,-4.1,0.0,RM,64.020179,10.35306
8780,2024-12-31 20:00:00+00:00,-5.3,0.0,RM,64.020179,10.35306
8781,2024-12-31 21:00:00+00:00,-5.8,0.0,RM,64.020179,10.35306
8782,2024-12-31 22:00:00+00:00,-6.4,0.0,RM,64.020179,10.35306
